In [171]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

# --- 1) Caminhos
DATA = Path("../data/interim")
DATA_ANALYSIS = Path("../data/analysis")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [172]:
# --- 2) Leitura e padronização de nomes (evita colisões no merge)
companies = pd.read_parquet(DATA / "ns_resolver/companies.parquet").drop(columns=["longBusinessSummary"])  # 1 linha por empresa (tem 'apex')

dns_status = (
    pd.read_parquet(DATA / "ns_resolver/dns_status.parquet")
      .rename(columns={"domain": "apex", "status": "dns_status"})
      .drop_duplicates("apex")
)

ns_hosts = (
    pd.read_parquet(DATA / "ns_resolver/ns_hosts.parquet")
      .rename(columns={"domain": "apex"})
      .drop_duplicates(["apex", "ns"])
)

ns_ips = (
    pd.read_parquet(DATA / "ns_resolver/ns_ips.parquet")
      .rename(columns={"rtype": "ip_type", "status": "ip_lookup_status"})
)

ip_asn = (
    pd.read_parquet(DATA / "asn_resolver/ip_asn.parquet")
      .rename(columns={"cc": "prefix_cc", "registry": "prefix_registry", "status": "asn_status"})
      .drop_duplicates("ip")
)

asn_org = (
    pd.read_parquet(DATA / "asn_resolver" / "asn_org_info.parquet")
      [["asn", "handle", "org", "country_code"]]
      .rename(columns={"handle": "as_handle", "org": "as_org", "country_code": "as_country_code"})
      .drop_duplicates("asn")
)

In [173]:
# --- 3) Encadeamento: empresa -> NS -> IP -> ASN -> organização
full = (
    companies
    .merge(dns_status, on="apex", how="left", validate="many_to_one")
    .merge(ns_hosts,   on="apex", how="left")
    .merge(ns_ips,     on="ns",   how="left")
    .merge(ip_asn,     on="ip",   how="left", validate="many_to_one")
    .merge(asn_org,    on="asn",  how="left", validate="many_to_one")
)

In [174]:
# --- 4) Status de DNS consolidado
full.loc[full["apex"].isna(), "dns_status"] = "no_website"
full["dns_status"] = full["dns_status"].fillna(
    full["ns"].notna().map({True: "ok", False: "not_resolved"})
)

In [175]:
# --- 5) Conferências
assert full["rank"].nunique() == companies["rank"].nunique(), "empresas perdidas no merge"
print(f"{len(companies)} empresas -> {len(full)} linhas")
print(full["dns_status"].value_counts())
print("linhas com IP sem ASN:", (full["ip"].notna() & full["asn"].isna()).sum())
print(full.loc[full["asn"].notna(), ["as_handle", "as_org", "as_country_code"]].isna().sum())


10635 empresas -> 83539 linhas
dns_status
ok            83163
no_website      343
no_ns            18
timeout          13
nxdomain          2
Name: count, dtype: int64
linhas com IP sem ASN: 13
as_handle          1
as_org             1
as_country_code    1
dtype: int64


In [176]:
# --- 6) Salva
full.to_parquet(DATA / "full_dns_enriched.parquet", index=False)
full.to_csv(DATA / "full_dns_enriched.csv", index=False)

In [177]:
from src.pipeline_audit import run_audit
cls = run_audit(out_path="../data/analysis/pipeline_limitations_report.txt")  # devolve um DataFrame com a etapa de falha por empresa

AUDITORIA DO PIPELINE - LIMITAÇÕES E EMPRESAS NÃO RESOLVIDAS
Gerado em 2026-10-13 13:26

1. FUNIL: TAMANHO DE CADA TABELA INTERMEDIÁRIA
------------------------------------------------------------------------------
  raw (top_companies_revenue_20261012.csv): 11345 linhas
  01_data_after_yfinance.csv           :   10635 linhas | 10635 empresas
  ns_resolver/companies.parquet        :   10635 linhas | 10635 empresas | 10292 com apex | 10233 apex únicos
  ns_resolver/dns_status.parquet       :      33 domínios com falha registrada
  ns_resolver/ns_hosts.parquet         :   31934 pares (domínio, NS) | 10200 domínios | 10768 NS únicos
  ns_resolver/ns_ips.parquet           :   26413 linhas | 10768 NS | 18862 IPs únicos
  asn_resolver/ip_asn.parquet          :   18862 IPs | 1424 ASNs
  asn_resolver/asn_org_info.parquet    :    1424 ASNs com org
  full_dns_enriched.parquet            :   83539 linhas | 10635 empresas

2. CONSISTÊNCIA ENTRE TABELAS (itens órfãos = informação perdida entre etap

In [ ]:
# --- 7) Salva só completos

# mantém só as linhas completamente resolvidas: NS -> IP -> ASN -> as_org

resolved_cols = ["ns", "ip", "asn", "as_org"]

full_resolved = full.dropna(subset=resolved_cols).copy()
full_unresolved = full.drop(full_resolved.index)  # o que foi removido, para auditoria

print(f"linhas: {len(full)} -> {len(full_resolved)} "
      f"(removidas {len(full_unresolved)}, {len(full_unresolved) / len(full):.1%})")
print(f"empresas: {full['rank'].nunique()} -> {full_resolved['rank'].nunique()}")
full_resolved.to_parquet(DATA / "full_resolved_clean.parquet", index=False)
full_resolved.to_csv(DATA / "full_resolved_clean.csv", index=False)
# salva a base limpa



linhas: 83539 -> 76403 (removidas 7136, 8.5%)
empresas: 10635 -> 10250


In [180]:
from src.depara_as_org import DEPARA_AS_ORG

# de-para -> DataFrame indexado por as_org
dp = pd.DataFrame.from_dict(DEPARA_AS_ORG, orient="index",
                            columns=["as_org_grupo", "as_country_code_grupo"])

# guarda os valores originais (auditoria)
full["as_org_original"] = full["as_org"]
full["as_country_code_original"] = full["as_country_code"]

# left join pelo nome da org; quem não está no de-para fica NaN
full = full.join(dp, on="as_org")

# aplica: usa o grupo/país controlador quando existe, senão mantém o original
full["as_org"] = full["as_org_grupo"].fillna(full["as_org"])
full["as_country_code"] = full["as_country_code_grupo"].fillna(full["as_country_code"])
full["as_org_remapped"] = full["as_org_grupo"].notna()   # flag: linha alterada pelo de-para

full = full.drop(columns=["as_org_grupo", "as_country_code_grupo"])

# conferência
print("linhas alteradas:", full["as_org_remapped"].sum(), f"({full['as_org_remapped'].mean():.1%})")
print("as_org únicas: antes", full["as_org_original"].nunique(), "-> depois", full["as_org"].nunique())
print("empresas com org remapeada:", full.loc[full["as_org_remapped"], "rank"].nunique())

full.to_parquet(DATA / "full_resolved_clean_depara.parquet", index=False)
full.to_csv(DATA / "full_resolved_clean_depara.csv", index=False)


linhas alteradas: 10 (0.0%)
as_org únicas: antes 1084 -> depois 1084
empresas com org remapeada: 5
